# RT Notebook 16: Projection-Driven Organizational DoF Isolation

Purpose: test whether increasing admissible organizational degree of freedom creates structurally novel continuation organizations while the primitive binding law remains unchanged.

This notebook is a bounded symbolic campaign scaffold. It is not theorem proof or external validation.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Sequence, Tuple

import networkx as nx
import pandas as pd

SPEC_ID = 'MPF_SIM_PROJECTION_DOF_ISOLATION_001'
SEED = 160016
DOF_VALUES = [1, 2, 3, 4, 5, 6]
MAX_DEPTH = 4
MAX_EXPANSIONS_PER_NODE = 3
OUTPUT_DIR = Path('/content') / SPEC_ID if Path('/content').exists() else Path('departments/colab/results') / SPEC_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR)


## Bounded Organizational Model

We keep one primitive binder fixed. Organizational DoF changes only the number of admissible child positions at each binding event.


In [ ]:
@dataclass(frozen=True)
class Org:
    children: Tuple['Org', ...] = ()

    @property
    def is_atom(self) -> bool:
        return len(self.children) == 0

ATOM = Org()

def signature(org: Org) -> str:
    if org.is_atom:
        return 'P'
    return 'RT(' + ','.join(signature(child) for child in org.children) + ')'

def canonical(children: Sequence[Org]) -> Tuple[Org, ...]:
    return tuple(sorted(children, key=signature))

def primitive_bind(children: Sequence[Org]) -> Org:
    return Org(children=canonical(children))

def max_depth(org: Org) -> int:
    if org.is_atom:
        return 1
    return 1 + max(max_depth(child) for child in org.children)

def node_count(org: Org) -> int:
    return 1 + sum(node_count(child) for child in org.children)

def primitive_preserved(org: Org) -> bool:
    if org.is_atom:
        return True
    return all(primitive_preserved(child) for child in org.children)

def closed_under_dof(org: Org, dof: int) -> bool:
    if org.is_atom:
        return dof == 1
    if len(org.children) != dof:
        return False
    return all(child.is_atom or closed_under_dof(child, dof) for child in org.children)

def symmetry_class(org: Org) -> str:
    if org.is_atom:
        return 'symmetric'
    child_sigs = [signature(child) for child in org.children]
    unique = len(set(child_sigs))
    if unique == 1:
        return 'symmetric'
    if unique == len(child_sigs):
        return 'asymmetric'
    return 'mixed'

def arity_profile(org: Org) -> Tuple[int, ...]:
    if org.is_atom:
        return (0,)
    values = [len(org.children)]
    for child in org.children:
        values.extend(arity_profile(child))
    return tuple(sorted(values))


In [ ]:
def lawful(org: Org, dof: int, depth_limit: int = MAX_DEPTH) -> bool:
    if max_depth(org) > depth_limit:
        return False
    if org.is_atom:
        return True
    if len(org.children) > dof:
        return False
    return all(lawful(child, dof, depth_limit) for child in org.children)

def generate_domain(dof: int, depth_limit: int = MAX_DEPTH):
    lawful_set = {ATOM}
    illegal = set()
    frontier = {ATOM}
    for _ in range(depth_limit):
        next_frontier = set()
        current = sorted(frontier, key=signature)
        base = sorted(lawful_set, key=signature)[: max(3, dof + 1)]
        for parent in current:
            for arity in range(1, dof + 1):
                pool = base[: max(1, min(len(base), arity + 1))]
                candidate = primitive_bind([parent, *pool[: max(0, arity - 1)]])
                if lawful(candidate, dof, depth_limit):
                    if candidate not in lawful_set:
                        lawful_set.add(candidate)
                        next_frontier.add(candidate)
                else:
                    illegal.add(candidate)
            if not parent.is_atom and len(parent.children) < dof:
                expanded = primitive_bind([*parent.children, ATOM])
                if lawful(expanded, dof, depth_limit):
                    if expanded not in lawful_set:
                        lawful_set.add(expanded)
                        next_frontier.add(expanded)
                else:
                    illegal.add(expanded)
        if not next_frontier:
            break
        frontier = next_frontier
    return sorted(lawful_set, key=signature), sorted(illegal, key=signature)

def one_step_successors(org: Org, dof: int):
    successors = set()
    if org.is_atom:
        successors.add(primitive_bind([ATOM]))
        if dof >= 2:
            successors.add(primitive_bind([ATOM, ATOM]))
    else:
        if len(org.children) < dof:
            successors.add(primitive_bind([*org.children, ATOM]))
        for idx, child in enumerate(org.children):
            for replacement in one_step_successors(child, dof):
                updated = list(org.children)
                updated[idx] = replacement
                successors.add(primitive_bind(updated))
    return sorted([s for s in successors if lawful(s, dof)], key=signature)[: MAX_EXPANSIONS_PER_NODE * max(1, dof)]

def build_graph(domain, dof: int) -> nx.DiGraph:
    graph = nx.DiGraph()
    keys = {signature(org): org for org in domain}
    for org in domain:
        sid = signature(org)
        graph.add_node(sid, depth=max_depth(org), nodes=node_count(org), symmetry=symmetry_class(org))
    for org in domain:
        sid = signature(org)
        for succ in one_step_successors(org, dof):
            tid = signature(succ)
            if tid in keys:
                graph.add_edge(sid, tid)
    return graph


## Projection and Novelty Tests


In [ ]:
def project_to_dof(org: Org, target_dof: int) -> Org:
    if org.is_atom or target_dof <= 1:
        return ATOM
    projected_children = [project_to_dof(child, target_dof) for child in org.children[:target_dof]]
    return primitive_bind(projected_children) if projected_children else ATOM

def lossless_projection_exists(org: Org, lower_domains, active_dof: int) -> bool:
    sig = signature(org)
    for target_dof in range(1, active_dof):
        projected = project_to_dof(org, target_dof)
        if signature(projected) == sig and sig in lower_domains[target_dof]:
            return True
    return False

def projection_loss_record(org: Org, active_dof: int, lower_domains):
    records = []
    src_sig = signature(org)
    for target_dof in range(1, active_dof):
        projected = project_to_dof(org, target_dof)
        records.append({
            'source_dof': active_dof,
            'target_dof': target_dof,
            'source_signature': src_sig,
            'projected_signature': signature(projected),
            'lossless': signature(projected) == src_sig,
            'closure_preserved': closed_under_dof(org, active_dof) == closed_under_dof(projected, target_dof),
            'symmetry_preserved': symmetry_class(org) == symmetry_class(projected),
            'depth_delta': max_depth(org) - max_depth(projected),
        })
    return records


In [ ]:
def entropy(counter: Counter) -> float:
    total = sum(counter.values())
    if total == 0:
        return 0.0
    value = 0.0
    for count in counter.values():
        p = count / total
        value -= p * math.log2(p)
    return value

def compute_metrics(dof: int, domain, illegal, graph: nx.DiGraph, lower_domains):
    lawful_rows = []
    novelty_rows = []
    projection_rows = []
    symmetry_counter = Counter()
    motif_counter = Counter()
    primitive_preserved_count = 0
    closure_count = 0
    depth_values = []
    branching_values = []
    for org in domain:
        sig = signature(org)
        cls = symmetry_class(org)
        symmetry_counter[cls] += 1
        motif_counter[(max_depth(org), arity_profile(org), cls)] += 1
        primitive_ok = primitive_preserved(org)
        primitive_preserved_count += int(primitive_ok)
        closed = closed_under_dof(org, dof)
        closure_count += int(closed)
        depth_values.append(max_depth(org))
        branching_values.append(graph.out_degree(sig))
        stable = graph.out_degree(sig) == 0
        novel = False if dof == 1 else not lossless_projection_exists(org, lower_domains, dof)
        lawful_rows.append({
            'dof': dof,
            'signature': sig,
            'depth': max_depth(org),
            'node_count': node_count(org),
            'symmetry_class': cls,
            'closed': closed,
            'stable': stable,
            'primitive_preserved': primitive_ok,
            'novel_to_lower_dof': novel,
        })
        if novel:
            novelty_rows.append({
                'dof': dof,
                'signature': sig,
                'depth': max_depth(org),
                'symmetry_class': cls,
                'stable': stable,
            })
        projection_rows.extend(projection_loss_record(org, dof, lower_domains))
    lawful_count = len(domain)
    metrics = {
        'dof': dof,
        'admissible_continuation_count': lawful_count,
        'illegal_organization_count': len(illegal),
        'closure_rate': closure_count / lawful_count if lawful_count else 0.0,
        'organization_entropy': entropy(motif_counter),
        'projection_diversity': len({row['projected_signature'] for row in projection_rows if row['target_dof'] < dof}),
        'primitive_preservation_rate': primitive_preserved_count / lawful_count if lawful_count else 0.0,
        'new_stable_organization_count': sum(1 for row in novelty_rows if row['stable']),
        'average_organization_depth': sum(depth_values) / len(depth_values) if depth_values else 0.0,
        'max_organization_depth': max(depth_values) if depth_values else 0,
        'average_continuation_branching': sum(branching_values) / len(branching_values) if branching_values else 0.0,
        'novel_organization_rate': len(novelty_rows) / lawful_count if lawful_count else 0.0,
        'symmetric_count': symmetry_counter['symmetric'],
        'asymmetric_count': symmetry_counter['asymmetric'],
        'mixed_count': symmetry_counter['mixed'],
    }
    return metrics, lawful_rows, novelty_rows, projection_rows


## Run the DoF Sweep and Write Recoverable Outputs


In [ ]:
domains = {}
lower_domain_maps = defaultdict(dict)
all_metrics = []
lawful_catalog = []
novel_catalog = []
projection_rows = []

for dof in DOF_VALUES:
    domain, illegal = generate_domain(dof, MAX_DEPTH)
    graph = build_graph(domain, dof)
    metrics, lawful_rows, novelty_rows, projection_loss_rows = compute_metrics(dof, domain, illegal, graph, lower_domain_maps)
    domains[dof] = {signature(org): org for org in domain}
    lower_domain_maps[dof] = domains[dof]
    all_metrics.append(metrics)
    lawful_catalog.extend(lawful_rows)
    novel_catalog.extend(novelty_rows)
    projection_rows.extend(projection_loss_rows)
    pd.DataFrame(lawful_rows).to_json(OUTPUT_DIR / f'lawful_catalog_dof_{dof}.json', orient='records', indent=2)
    pd.DataFrame(novelty_rows).to_json(OUTPUT_DIR / f'novel_catalog_dof_{dof}.json', orient='records', indent=2)

metrics_df = pd.DataFrame(all_metrics)
lawful_df = pd.DataFrame(lawful_catalog)
novel_df = pd.DataFrame(novel_catalog)
projection_df = pd.DataFrame(projection_rows)

metrics_df.to_csv(OUTPUT_DIR / 'dof_metrics.csv', index=False)
lawful_df.to_json(OUTPUT_DIR / 'lawful_organization_catalog.jsonl', orient='records', lines=True)
novel_df.to_json(OUTPUT_DIR / 'novel_organization_catalog.json', orient='records', indent=2)
projection_df.to_csv(OUTPUT_DIR / 'projection_loss_table.csv', index=False)

display(metrics_df)


In [ ]:
manifest = {
    'spec_id': SPEC_ID,
    'status': 'NOT_EXECUTED_IN_SOURCE_NOTEBOOK',
    'claim_ceiling': 'C1_SPECIFICATION_ONLY',
    'seed': SEED,
    'dof_values': DOF_VALUES,
    'max_depth': MAX_DEPTH,
    'output_files': {
        'dof_metrics': 'dof_metrics.csv',
        'lawful_catalog': 'lawful_organization_catalog.jsonl',
        'novel_catalog': 'novel_organization_catalog.json',
        'projection_loss_table': 'projection_loss_table.csv',
    },
    'follow_on_campaign': 'MPF_SIM_PROJECTION_INDUCTION_001',
    'non_claims': [
        'No theorem or ontology promotion from notebook source alone.',
        'No external physical validation.',
        'No claim above bounded computational evidence after result induction.',
    ],
}
with open(OUTPUT_DIR / 'manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))


## Interpretation Boundary

Notebook 16 can only support bounded evidence about this declared symbolic organizational model after execution outputs are recovered and governed. It does not prove that higher organizational DoF changes ontology or that the full calculus admits the same novelty structure.

Follow-on governed campaign: `MPF_SIM_PROJECTION_INDUCTION_001`.
